# Introduction

Final Project ini membahas pengembangan aplikasi chatbot informasi Program Jaminan Kesehatan Nasional (JKN) yang digunakan untuk memberikan informasi terkait regulasi dan ketentuan JKN berdasarkan peraturan perundang-undangan yang berlaku. Aplikasi chatbot dirancang agar pengguna dapat melakukan tanya jawab secara langsung mengenai informasi JKN melalui antarmuka berbasis web.

Pada pengembangan sistem ini, teknologi Artificial Intelligence (AI) yang digunakan adalah model Gemini versi gemini-3.1-flash-lite. Model ini digunakan untuk memproses pertanyaan pengguna dan menghasilkan jawaban yang relevan berdasarkan konteks informasi mengenai Program JKN. Penggunaan model Gemini dipilih karena memiliki performa yang ringan dan respons yang cepat untuk kebutuhan chatbot berbasis teks.

# Project Chatbot

In [37]:
!pip install streamlit pymupdf sentence-transformers scikit-learn google-genai pyngrok

In [38]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
from pyngrok import ngrok
from google.colab import userdata

ngrok.set_auth_token(userdata.get('NGROK_TOKEN'))
print("ngrok token berhasil dikonfigurasi!")

ngrok token berhasil dikonfigurasi!


In [40]:
import os
from google.colab import userdata

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI1")
print("Secret berhasil dimuat ke environment.")

Secret berhasil dimuat ke environment.


In [41]:
import subprocess
import time

def run_streamlit(filename, port=8501):
    # Kill SEMUA proses streamlit, bukan hanya yang kita spawn
    subprocess.run(["pkill", "-f", "streamlit"], capture_output=True)

    # Force-free port kalau masih ada yang nempel
    subprocess.run(["fuser", "-k", f"{port}/tcp"], capture_output=True)

    # Tutup semua tunnel ngrok
    ngrok.kill()

    # Tunggu port benar-benar bebas
    time.sleep(3)

    proc = subprocess.Popen(
        [
            "streamlit", "run", filename,
            "--server.headless=true",
            "--server.port", str(port),
            "--server.enableCORS=false",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(3)

    public_url = ngrok.connect(port)
    print(f"Streamlit berjalan: {public_url}")

    return proc

In [42]:
%%writefile JKN_Chat_Bot_app.py
# LIBRARY
import os
import fitz
import numpy as np
import streamlit as st

from google import genai
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# CONFIG
FAQ_FOLDER = "/content/drive/MyDrive/Colab Notebooks/Belajar AI/Final Project/faq_docs"
google_api_key = os.environ.get("GEMINI_API_KEY", "")

# PAGE
st.set_page_config(
    page_title="Chatbot Informasi Program JKN",
    layout="centered"
)

# SIDEBAR
with st.sidebar:
    st.subheader("Pengaturan")

    theme = st.selectbox(
        "Mode Tampilan",
        ["System", "Light", "Dark"]
    )

    reset_button = st.button("Reset Percakapan", use_container_width=True)

    st.divider()
    st.subheader("Dokumen Aktif")

    if os.path.exists(FAQ_FOLDER):
        pdf_files = sorted([f for f in os.listdir(FAQ_FOLDER) if f.endswith(".pdf")])
        if pdf_files:
            for f in pdf_files:
                st.caption(f" {f}")
            st.caption(f"\n Total: {len(pdf_files)} dokumen")
        else:
            st.warning("Tidak ada file PDF di folder tersebut.")
    else:
        st.error(f"Folder tidak ditemukan:\n`{FAQ_FOLDER}`")
        st.info("Pastikan Google Drive sudah di-mount dan path sudah benar.")

# VALIDATE API KEY
if not google_api_key:
    st.info("Masukkan Google AI API Key di sidebar untuk mulai chat.", icon="🗝️")
    st.stop()

# THEME
if theme == "Dark":
    st.markdown(
        """
        <style>
        .stApp { background-color: #0e1117; color: white; }
        </style>
        """,
        unsafe_allow_html=True
    )
elif theme == "Light":
    st.markdown(
        """
        <style>
        .stApp { background-color: #ffffff; color: black; }
        </style>
        """,
        unsafe_allow_html=True
    )

# HEADER
st.title("Chatbot Informasi Program JKN")
st.caption("Jawaban berdasarkan Peraturan Perundang-undangan JKN - BPJS Kesehatan")
st.divider()

# GEMINI CLIENT
if (
    "genai_client" not in st.session_state
    or getattr(st.session_state, "_last_key", None) != google_api_key
):
    try:
        st.session_state.genai_client = genai.Client(api_key=google_api_key)
        st.session_state._last_key = google_api_key
        st.session_state.pop("messages", None)
    except Exception as e:
        st.error(f"API Key tidak valid: {e}")
        st.stop()

# LOAD SEMUA PDF DARI FOLDER DRIVE
def load_all_pdfs(folder_path):
    """
    Membaca semua file PDF dari folder Google Drive.
    Setiap chunk disimpan beserta nama file sumbernya.
    """
    all_chunks  = []
    all_sources = []

    if not os.path.exists(folder_path):
        st.error(
            f"Folder `{folder_path}` tidak ditemukan. "
            "Pastikan Google Drive sudah di-mount."
        )
        return [], []

    pdf_files = sorted([f for f in os.listdir(folder_path) if f.endswith(".pdf")])

    if not pdf_files:
        st.warning("Tidak ada file PDF di folder tersebut.")
        return [], []

    for filename in pdf_files:
        filepath = os.path.join(folder_path, filename)
        try:
            doc       = fitz.open(filepath)
            full_text = ""

            for page in doc:
                text = page.get_text()
                if text.strip():
                    full_text += text + "\n"

            doc.close()

            chunk_size = 1000
            overlap    = 100
            step       = chunk_size - overlap

            for i in range(0, len(full_text), step):
                chunk = full_text[i : i + chunk_size]
                if chunk.strip():
                    all_chunks.append(chunk)
                    all_sources.append(filename)

        except Exception as e:
            st.warning(f"Gagal membaca `{filename}`: {e}")

    return all_chunks, all_sources


# EMBEDDING MODEL
@st.cache_resource
def load_embedding_model():
    return SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

@st.cache_resource
def build_faq_index():
    """
    Membangun index embedding dari semua PDF di Google Drive.
    Di-cache agar tidak di-rebuild setiap kali ada interaksi.
    """
    chunks, sources = load_all_pdfs(FAQ_FOLDER)

    if not chunks:
        return [], [], []

    model      = load_embedding_model()
    embeddings = model.encode(
        chunks,
        show_progress_bar=False,
        batch_size=32
    )

    return chunks, embeddings, sources

# RETRIEVE
def retrieve_context(
    question,
    chunks,
    embeddings,
    sources,
    threshold=0.20,
    top_k=3
):
    """
    Mengambil top_k chunk paling relevan dari semua dokumen.
    Mengembalikan gabungan konteks + daftar sumber unik.
    """
    if not chunks:
        return None, 0.0, []

    model       = load_embedding_model()
    q_embedding = model.encode([question])
    sims        = cosine_similarity(q_embedding, embeddings)[0]
    top_indices = np.argsort(sims)[::-1][:top_k]

    if sims[top_indices[0]] < threshold:
        return None, float(sims[top_indices[0]]), []

    context_parts = []
    used_sources  = []

    for idx in top_indices:
        if sims[idx] >= threshold:
            context_parts.append(chunks[idx])
            src = sources[idx]
            if src not in used_sources:
                used_sources.append(src)

    combined_context = "\n\n---\n\n".join(context_parts)
    return combined_context, float(sims[top_indices[0]]), used_sources


# INTENT
def detect_intent(question):
    q = question.lower()

    admin_keywords = [
        "ubah data", "perubahan data", "cek peserta",
        "status peserta", "kepesertaan", "data peserta",
        "daftar peserta", "pendaftaran"
    ]

    complaint_keywords = [
        "kenapa tidak ditanggung", "tidak ditanggung",
        "keluhan", "komplain", "pengaduan", "tidak terlayani"
    ]

    if any(k in q for k in admin_keywords):
        return "admin"
    if any(k in q for k in complaint_keywords):
        return "complaint"
    return "faq"


# RESPONSE
def generate_answer(question, context=None):
    if context:
        prompt = f"""
Anda adalah chatbot resmi informasi Program JKN BPJS Kesehatan Indonesia.
Sumber jawaban utama Anda adalah peraturan perundang-undangan JKN.

Gunakan konteks peraturan berikut sebagai dasar jawaban.
Jika konteks belum menjawab secara lengkap, Anda boleh menambahkan
informasi umum yang relevan tentang JKN/BPJS Kesehatan.

Jawablah dengan bahasa Indonesia yang jelas, ringkas, dan mudah dipahami.
Gunakan format poin jika jawaban lebih dari satu poin.

Konteks Peraturan:
{context}

Pertanyaan:
{question}
"""
    else:
        prompt = f"""
Anda adalah chatbot resmi informasi Program JKN BPJS Kesehatan Indonesia.

Pertanyaan berikut tidak ditemukan secara spesifik dalam dokumen peraturan
yang tersedia. Jawablah berdasarkan pengetahuan umum Anda tentang program
JKN dan BPJS Kesehatan Indonesia.

Jika benar-benar tidak mengetahui jawabannya, sampaikan dengan jujur dan
arahkan pengguna menghubungi BPJS Care Center di nomor 165.

Jawablah dengan bahasa Indonesia yang jelas dan mudah dipahami.

Pertanyaan:
{question}
"""

    response = st.session_state.genai_client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt,
        config={
            "temperature" : 0.15,
            "top_k"       : 30,
            "top_p"       : 0.8,
            "max_output_tokens": 800
        }
    )

    return response.text


# INDEX
chunks, embeddings, sources = build_faq_index()

# SESSION STATE
if "messages" not in st.session_state:
    st.session_state.messages = []

if reset_button:
    st.session_state.pop("messages", None)
    st.rerun()

# HISTORY CHAT
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# INPUT RESPONSE
prompt = st.chat_input("Tanyakan seputar peraturan Program JKN...")

# DIALOG
if prompt:

    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    intent      = detect_intent(prompt)
    used_sources = []

    if intent == "admin":
        answer = (
            "Untuk keperluan **perubahan data peserta** atau "
            "**pengecekan status kepesertaan**, silakan gunakan:\n\n"
            "- **Mobile JKN** (App Store / Play Store)\n"
            "- **PANDAWA** (Pelayanan Administrasi melalui WhatsApp)\n"
            "- **Website BPJS Kesehatan**: bpjs-kesehatan.go.id\n"
            "- **BPJS Care Center**: 165"
        )

    elif intent == "complaint":
        context, skor, used_sources = retrieve_context(
            "pelayanan yang tidak ditanggung JKN",
            chunks, embeddings, sources
        )

        if context:
            answer = (
                generate_answer("Apa saja pelayanan yang tidak ditanggung JKN?", context)
                + "\n\n *Untuk kasus spesifik, sampaikan pengaduan melalui "
                "kanal resmi BPJS Kesehatan atau hubungi **165**.*"
            )
        else:
            answer = (
                "Untuk pertanyaan mengenai pelayanan yang tidak ditanggung, "
                "chatbot hanya dapat memberikan informasi umum dari peraturan.\n\n"
                "Silakan hubungi **BPJS Care Center 165** untuk penanganan "
                "kasus spesifik Anda."
            )

    # ── Intent: FAQ umum (RAG)
    else:
        context, skor, used_sources = retrieve_context(
            prompt, chunks, embeddings, sources
        )
        answer = generate_answer(prompt, context)

    # Tambahkan referensi sumber dokumen jika ada
    if used_sources:
        source_list = "\n".join([f"  -  {s}" for s in used_sources])
        answer += f"\n\n---\n *Sumber dokumen:*\n{source_list}"

    # Tampilkan jawaban
    with st.chat_message("assistant"):
        st.markdown(answer)

    # Simpan ke riwayat
    st.session_state.messages.append({"role": "assistant", "content": answer})


Overwriting JKN_Chat_Bot_app.py


In [47]:
proc = run_streamlit("JKN_Chat_Bot_app.py")

Streamlit berjalan: NgrokTunnel: "https://primal-google-facecloth.ngrok-free.dev" -> "http://localhost:8501"


In [46]:
# Hentikan Streamlit
try:
    proc.terminate()
    print("Streamlit dihentikan.")
except:
    print("Tidak ada proses yang berjalan.")

# Tutup semua tunnel ngrok
ngrok.kill()
print("Tunnel ngrok ditutup.")

Streamlit dihentikan.
Tunnel ngrok ditutup.


# Conclusion

Aplikasi ini sudah bisa berjalan sesuai dengan keinginan dan jawaban sesuai dengan peraturan perundang-undangan yang dinput. Walaupun pada apliaksi ini banyak kekurangan dari tampilan awal dan masih banyak code yan perlu di update.